# FlowGuard -- Keras/TensorFlow model (LinuxONE)

Trains a neural network on the features exported by
`01_prepare_and_baseline.ipynb`, then scores it through the **same**
`flowguard.evaluation.metrics.evaluate()` the XGBoost experiments use, so the
numbers are directly comparable.

**Run `01_prepare_and_baseline.ipynb` first** -- this notebook needs its
`*_features.h5`. Nothing here recomputes features, so iterating on the model
costs minutes, not the full extraction run.

Uses only what the VM already has: `tensorflow 2.9.3` / `keras 2.9.0`,
`h5py 3.8.0`, `scikit-learn 1.4.0`, `numpy 1.24.4`, `matplotlib`, `seaborn`.
Nothing to install.

### Three things that would silently ruin this, handled below
1. **NaNs.** GFP leaves self-transfer rows as NaN by design. XGBoost handles
   that natively; Keras does not -- one NaN gives you a NaN loss and a dead
   model on the first epoch. Imputed here, after scaling.
2. **Scale.** Trees are scale-invariant, networks are not. `StandardScaler` is
   fitted on the **training partition only**, in chunks.
3. **Scoring.** `tf.keras.metrics.AUC` approximates over ~200 histogram
   buckets, which at a ~0.09% base rate is far too coarse to trust. It is used
   only for early stopping; every reported number comes from the project's own
   `evaluate()`.


In [ ]:
import os
import sys
import platform

# 2 vCPUs -- tell TF that up front instead of letting it oversubscribe.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(max(1, os.cpu_count() or 1))
tf.config.threading.set_inter_op_parallelism_threads(1)

print("python    :", sys.version.split()[0])
print("machine   :", platform.machine(), "/ byte order:", sys.byteorder)
print("tensorflow:", tf.__version__)
print("keras     :", tf.keras.__version__)
print("GPUs      :", tf.config.list_physical_devices("GPU") or "none (expected)")


## 1. Locate the exported features and the flowguard source

In [ ]:
from pathlib import Path
import h5py

HOME, CWD = Path.home(), Path.cwd()

# ---- Optional overrides (leave None to auto-detect) -----------------------
H5_OVERRIDE = None             # e.g. Path("/home/linux1/flowguard_outputs/HI-Small_features.h5")
FLOWGUARD_SRC_OVERRIDE = None  # e.g. Path("/home/linux1/flowguard/src")
# ---------------------------------------------------------------------------


def _find(candidates, probe):
    for c in candidates:
        if probe(c):
            return c
    return None


FLOWGUARD_SRC = (
    Path(FLOWGUARD_SRC_OVERRIDE) if FLOWGUARD_SRC_OVERRIDE
    else Path(os.environ["FLOWGUARD_SRC"]) if os.environ.get("FLOWGUARD_SRC")
    else _find(
        [*[p / "src" for p in (CWD, *CWD.parents)],
         HOME / "flowguard" / "src", HOME / "Project" / "flowguard" / "src"],
        lambda c: (c / "flowguard" / "data" / "schema.py").exists())
)
if FLOWGUARD_SRC is None:
    raise FileNotFoundError(
        "Could not find the flowguard 'src' directory. Set FLOWGUARD_SRC_OVERRIDE "
        "above, or export FLOWGUARD_SRC."
    )
sys.path.insert(0, str(FLOWGUARD_SRC))

OUTPUT_DIR = Path(os.environ.get("FLOWGUARD_OUTPUT_DIR", HOME / "flowguard_outputs"))
H5_PATH = (
    Path(H5_OVERRIDE) if H5_OVERRIDE
    else _find(sorted(OUTPUT_DIR.glob("*_features.h5")) + sorted(CWD.glob("data/*_features.h5")),
               lambda c: c.exists())
)
if H5_PATH is None:
    raise FileNotFoundError(
        f"No *_features.h5 found in {OUTPUT_DIR} or {CWD / 'data'}.\n"
        "Run 01_prepare_and_baseline.ipynb first -- its final section writes it."
    )

with h5py.File(H5_PATH, "r") as h5:
    N_TRAIN = int(h5.attrs["n_train"])
    N_VAL = int(h5.attrs["n_val"])
    N_TEST = int(h5.attrs["n_test"])
    N_FEATURES = int(h5.attrs["n_features"])
    FEATURE_NAMES = [n.decode() if isinstance(n, bytes) else str(n)
                     for n in h5["feature_names"][:]]
    GFP_FEATURE_SOURCE = str(h5.attrs.get("gfp_feature_source", "unknown"))
    y_all = h5["y"][:].astype(int)

TRAIN = (0, N_TRAIN)
VAL = (N_TRAIN, N_TRAIN + N_VAL)
TEST = (N_TRAIN + N_VAL, N_TRAIN + N_VAL + N_TEST)
y_train, y_val, y_test = (y_all[a:b] for a, b in (TRAIN, VAL, TEST))

print("features file:", H5_PATH)
print(f"  {N_FEATURES} features, source: {GFP_FEATURE_SOURCE}")
print(f"  train {N_TRAIN:,} ({y_train.sum():,} pos)   "
      f"val {N_VAL:,} ({y_val.sum():,} pos)   "
      f"test {N_TEST:,} ({y_test.sum():,} pos)")
print(f"  test base rate: {y_test.mean():.4%}")


## 2. Scaling

`StandardScaler.partial_fit` over the training partition in chunks, so the
full matrix is never resident. sklearn ignores NaNs when computing the
statistics and preserves them through `transform`; they are imputed to 0
afterwards, which -- post-scaling -- means "the training mean".


In [ ]:
from sklearn.preprocessing import StandardScaler

CHUNK = 200_000
scaler = StandardScaler()
with h5py.File(H5_PATH, "r") as h5:
    for start in range(TRAIN[0], TRAIN[1], CHUNK):
        stop = min(start + CHUNK, TRAIN[1])
        scaler.partial_fit(h5["X"][start:stop])
        print(f"  fitted {stop - TRAIN[0]:>10,} / {N_TRAIN:,} train rows", end="\r", flush=True)

# A zero-variance column would divide by ~0 and produce infinities.
scaler.scale_ = np.where(scaler.scale_ > 0, scaler.scale_, 1.0)
print(f"\nscaler fitted on {N_TRAIN:,} training rows only")


## 3. Data generator and model

Batches are read straight from HDF5, so peak RAM is one batch rather than the
whole partition.

Shuffling is at **batch granularity** (the order of contiguous blocks is
permuted each epoch), not row granularity -- row-level shuffling would mean
random-access reads into HDF5, which is far slower. Rows inside one batch are
therefore temporally adjacent. If training looks unstable, shrink
`BATCH_SIZE`; that is the knob this trade-off leaves you.


In [ ]:
import random

BATCH_SIZE = 4096


class H5Batches(tf.keras.utils.Sequence):
    """Streams (X, y) batches out of the HDF5 export, scaled and NaN-free."""

    def __init__(self, path, span, batch_size=BATCH_SIZE, scaler=None, shuffle=False):
        self.path, self.scaler, self.shuffle = str(path), scaler, shuffle
        start, stop = span
        self.blocks = [(s, min(s + batch_size, stop)) for s in range(start, stop, batch_size)]
        self._handle = None

    @property
    def _h5(self):
        if self._handle is None:          # opened lazily, once per worker
            self._handle = h5py.File(self.path, "r")
        return self._handle

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, i):
        lo, hi = self.blocks[i]
        X = self._h5["X"][lo:hi]
        y = self._h5["y"][lo:hi]
        if self.scaler is not None:
            X = self.scaler.transform(X)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
        return X, y.astype("float32")

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.blocks)


train_gen = H5Batches(H5_PATH, TRAIN, scaler=scaler, shuffle=True)
val_gen = H5Batches(H5_PATH, VAL, scaler=scaler)

tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_FEATURES,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
             tf.keras.metrics.AUC(name="roc_auc")],
)
model.summary()

# Mirrors XGBoost's scale_pos_weight = negatives / positives.
pos = int(y_train.sum())
class_weight = {0: 1.0, 1: (len(y_train) - pos) / max(pos, 1)}
print(f"\nclass_weight for the positive class: {class_weight[1]:,.0f}")


## 4. Train

Early stopping watches validation PR-AUC -- the validation partition is the
only thing allowed to influence when training stops. Test is never touched
here.


In [ ]:
import time

EPOCHS = 30

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=5,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5,
                                         patience=3, min_lr=1e-5, verbose=1),
]

started = time.perf_counter()
history = model.fit(
    train_gen, validation_data=val_gen, epochs=EPOCHS, class_weight=class_weight,
    callbacks=callbacks, workers=1, use_multiprocessing=False, verbose=2,
)
print(f"\ntrained in {time.perf_counter() - started:.0f}s "
      f"over {len(history.history['loss'])} epochs")


## 5. Score it the project's way

Predictions are calibrated with isotonic regression fitted on validation --
the same treatment `flowguard.models.xgb.XGBModel` gives its output, so the
scores mean the same thing -- then scored with the shared `evaluate()`.


In [ ]:
from sklearn.isotonic import IsotonicRegression
from flowguard.evaluation.metrics import evaluate, per_group_recall


def predict_span(span):
    """Predict a partition in batches, never materialising it whole."""
    gen = H5Batches(H5_PATH, span, scaler=scaler)
    return model.predict(gen, workers=1, use_multiprocessing=False, verbose=0).ravel()


raw_val, raw_test = predict_span(VAL), predict_span(TEST)

calibrator = IsotonicRegression(out_of_bounds="clip").fit(raw_val, y_val)
dnn_val_scores, dnn_test_scores = calibrator.predict(raw_val), calibrator.predict(raw_test)

dnn_val = evaluate(y_val, dnn_val_scores)
dnn_test = evaluate(y_test, dnn_test_scores)
print("Keras DNN -- test:")
print(dnn_test.summary())


## 6. Compare against the tree models

Picks up `results.json` from `01_prepare_and_baseline.ipynb` if it is there.
Gradient-boosted trees are the strong baseline on tabular data like this, so
do not be surprised if the network loses -- that is a result worth reporting,
not a bug to hide.


In [ ]:
import json
import pandas as pd

rows = {"DNN (Keras)": dnn_test.to_metadata()}
results_path = OUTPUT_DIR / "results.json"
if results_path.exists():
    prior = json.loads(results_path.read_text(encoding="utf-8"))
    for key, label in (("E0", "E0 (rules)"), ("E1", "E1 (transaction)"),
                       ("E2", f"E2 ({prior.get('E2', {}).get('feature_source', 'graph')})")):
        if key in prior:
            rows[label] = prior[key]["test"]
else:
    print(f"(no {results_path} -- run 01 for the XGBoost comparison)")


def _budget(metadata, budget, field):
    return next(b[field] for b in metadata["budgets"] if b["budget"] == budget)


comparison = pd.DataFrame(
    {
        "PR-AUC": {k: v["pr_auc"] for k, v in rows.items()},
        "ROC-AUC": {k: v["roc_auc"] for k, v in rows.items()},
        "lift": {k: v["lift_over_base_rate"] for k, v in rows.items()},
        "recall@1%": {k: _budget(v, 0.01, "recall") for k, v in rows.items()},
        "precision@1%": {k: _budget(v, 0.01, "precision") for k, v in rows.items()},
    }
).sort_values("PR-AUC", ascending=False)
print(comparison.round(4).to_string())


## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("binary cross-entropy")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history.history["pr_auc"], label="train")
axes[1].plot(history.history["val_pr_auc"], label="validation")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("PR-AUC (Keras approximation)")
axes[1].set_title("PR-AUC during training"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_training_history.png")
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(7, 6))
precision, recall, _ = precision_recall_curve(y_test, dnn_test_scores)
ax.plot(recall, precision, color="#937860",
        label=f"DNN (Keras)  PR-AUC={dnn_test.pr_auc:.4f}")

e2_score_path = OUTPUT_DIR / "e2_test_scores.npy"
if e2_score_path.exists():
    e2_scores = np.load(e2_score_path)
    if len(e2_scores) == len(y_test):
        e2_precision, e2_recall, _ = precision_recall_curve(y_test, e2_scores)
        e2_label = next((f"E2 (XGBoost)  PR-AUC={v['pr_auc']:.4f}"
                         for k, v in rows.items() if k.startswith("E2")), "E2 (XGBoost)")
        ax.plot(e2_recall, e2_precision, color="#C44E52", label=e2_label)

ax.axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1,
           label=f"base rate ({y_test.mean():.4%})")
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("Test set -- precision/recall")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_pr_curve.png")
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

# Colour is the fraction within each true class; the annotation is the raw
# count. A plain 2x2 at a ~0.09% base rate is all true negatives otherwise.
val_threshold = dnn_val.best_f1_threshold
budget_threshold = dnn_test.at_budget(0.01).threshold

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (title, thr) in zip(axes, [
    (f"best-F1 threshold from validation\n(score >= {val_threshold:.4f})", val_threshold),
    (f"1% alert budget\n(top {int(round(len(y_test) * 0.01)):,} scores)", budget_threshold),
]):
    cm = confusion_matrix(y_test, (dnn_test_scores >= thr).astype(int), labels=[0, 1])
    sns.heatmap(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), annot=cm, fmt=",d",
                cmap="Oranges", cbar=False, ax=ax,
                xticklabels=["predicted clean", "predicted laundering"],
                yticklabels=["actually clean", "actually laundering"])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{title}\nrecall {tp / max(tp + fn, 1):.1%}  |  "
                 f"precision {tp / max(tp + fp, 1):.2%}  |  {fp:,} false alerts", fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_confusion_matrices.png")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(x=dnn_test_scores[y_test == 0], stat="density", bins=60,
             color="#4C72B0", label="legitimate", alpha=0.6, ax=ax)
sns.histplot(x=dnn_test_scores[y_test == 1], stat="density", bins=60,
             color="#C44E52", label="laundering", alpha=0.6, ax=ax)
ax.set_xlabel("calibrated DNN score")
ax.set_title("DNN score distribution by class (test set)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_score_distribution.png")
plt.show()


In [ ]:
# Recall per laundering typology @1% budget
with h5py.File(H5_PATH, "r") as h5:
    test_typologies = np.array([
        t.decode() if isinstance(t, bytes) else str(t)
        for t in h5["pattern_type"][TEST[0]:TEST[1]]
    ])

typology_recall = per_group_recall(y_test, dnn_test_scores, test_typologies, budget=0.01)
if typology_recall:
    typ_df = pd.DataFrame(typology_recall).T.sort_values("recall")
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(typ_df))))
    ax.barh(typ_df.index, typ_df["recall"], color="#937860")
    ax.set_xlabel("recall @ 1% budget")
    ax.set_title("DNN -- recall by laundering typology")
    for y_pos, (_, row) in enumerate(typ_df.iterrows()):
        ax.text(row["recall"] + 0.01, y_pos,
                f"{int(row['caught'])}/{int(row['positives'])}", va="center")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "dnn_typology_recall.png")
    plt.show()
else:
    print("No typology labels in the test partition -- skipping.")


## 8. Save

Keras 2.9 predates the `.keras` format, so the model is written as HDF5.
The scaler goes out as plain `.npy` arrays rather than a joblib pickle: a
pickle written here would be tied to this exact sklearn/numpy build, and on a
big-endian machine that is a portability problem waiting to happen.


In [ ]:
model.save(OUTPUT_DIR / "dnn_model.h5")
np.save(OUTPUT_DIR / "dnn_scaler_mean.npy", scaler.mean_)
np.save(OUTPUT_DIR / "dnn_scaler_scale.npy", scaler.scale_)
np.save(OUTPUT_DIR / "dnn_test_scores.npy", dnn_test_scores)

dnn_results = {
    "model": "Keras DNN (128-64-32, dropout, isotonic-calibrated)",
    "tensorflow": tf.__version__,
    "machine": platform.machine(),
    "feature_source": GFP_FEATURE_SOURCE,
    "n_features": N_FEATURES,
    "epochs_run": len(history.history["loss"]),
    "class_weight_positive": class_weight[1],
    "val": dnn_val.to_metadata(),
    "test": dnn_test.to_metadata(),
    "typology_recall_at_1pct": typology_recall,
}
(OUTPUT_DIR / "dnn_results.json").write_text(
    json.dumps(dnn_results, indent=2, default=str), encoding="utf-8")
comparison.to_csv(OUTPUT_DIR / "dnn_comparison.csv")

print("wrote:")
for name in ("dnn_model.h5", "dnn_scaler_mean.npy", "dnn_scaler_scale.npy",
             "dnn_test_scores.npy", "dnn_results.json", "dnn_comparison.csv"):
    print("  ", name)


## 9. Self-check

In [ ]:
assert not np.isnan(dnn_test.pr_auc), (
    "DNN PR-AUC is NaN -- almost always NaN features reaching the network. "
    "Check the nan_to_num step in H5Batches.__getitem__."
)
assert dnn_test.pr_auc > y_test.mean(), (
    f"DNN PR-AUC {dnn_test.pr_auc:.5f} is at or below the base rate "
    f"{y_test.mean():.5f} -- the model learned nothing."
)
assert len(dnn_test_scores) == len(y_test), "score/label length mismatch"
assert np.isfinite(dnn_test_scores).all(), "non-finite scores produced"

print("All self-checks passed.\n")
print(comparison.round(4).to_string())
